💡 **Environment:** `clamp-analyses`

# Description

**Sandbox NB 04 — multi-tissue / top-N recompute + aggregation (gene-based + module ARCHS4).**

Phase 1 (NB00–03) worked on one tissue (Liver). This notebook extends the null adjustment to the
**full evaluation scope** — 49 tissues × 5 top-N thresholds, mapped to the **685-pair**
PharmacotherapyDB universe (the scope behind the published AUROCs: gene ≈ 0.583, ARCHS4 ≈ 0.625) —
so we can ask whether adjustment changes the headline numbers (NB05 does the inference).

Per tissue × threshold we recompute **four per-cell scorings on the same masked vectors**, then run
each through the *identical* NB10 downstream (rank → inner-merge gold standard → mean over thresholds
→ max over tissues), so only the per-cell score differs. They form a normalization ladder that
isolates what the raw dot product's magnitude contributes:

- **raw** — `−Lᵀ X_masked` (the pipeline score). Keeps both vector norms **and** the mean.
- **cosine** — `−(L_d · x)/(‖L_d‖‖x‖)`. Removes the **norms**, keeps the mean. This is the clean test
  of NEXT_STEPS concern #2 ("is the signal just magnitude?"): cosine strips exactly the norm and
  nothing else.
- **pearson** — `−corr(L_d, x_masked)`, centered over all genes/LVs. Removes the norms **and** the
  mean (= cosine of the mean-centered vectors). It is the analytic, scalable form of the
  permute-disease null (NB01 showed corr 0.999; exact under masking). A B=200 permutation
  **spot-check** at the bottom reconfirms `NES_permute ≈ pearson` at this scale.
- **background** — per-drug z across all ~364 DOIDs of raw's full matrix (competitive null).

cosine and pearson differ only by the mean-centering term, so comparing them (NB05) measures how much
of any normalization effect is norm-removal vs mean-removal.

Validation: recomputed **raw** reproduces the published 0.583 / 0.625. See `CLAUDE.md`.

# Modules loading

In [1]:
import sys
import json
import time
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

from pyprojroot import here

sys.path.insert(0, str(here('libs')))
from drug_disease_utils import map_traits_to_doid

/home/miltondp/software/miniforge3/envs/clamp-analyses/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Settings

In [2]:
SEED = 42
N_TISSUES = 49
SPOTCHECK_BPERM = 200       # permutation spot-check only

# top-N thresholds (reuse signif_test METHOD_THRESHOLDS; None == all, no masking)
METHOD_THRESHOLDS = {
    'gene_based':          [None, 50, 100, 250, 500],
    'module_based_archs4': [None, 5, 10, 25, 50],
}
METHODS = list(METHOD_THRESHOLDS)
SCORINGS = ['raw', 'cosine', 'pearson', 'background']

DATA_DIR = here('data/drug_disease_associations')
OUTPUT_DIR = here(
    'output/03_model_biology/00_archs4/02_drug_disease_associations/null_adjust_test')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LINCS_RAW_FILE = DATA_DIR / 'lincs-data.pkl'
LINCS_PROJ_FILE = here('output/03_model_biology/00_archs4/02_drug_disease_associations/'
                       '01_lincs_projection_archs4/lincs/lincs-projection.pkl')
SPREDIXCAN_RAW_DIR = here('output/03_model_biology/00_archs4/02_drug_disease_associations/'
                          '00_spredixcan_projection_archs4/spredixcan/raw')
SPREDIXCAN_PROJ_DIR = here('output/03_model_biology/00_archs4/02_drug_disease_associations/'
                           '00_spredixcan_projection_archs4/spredixcan/proj')

# Load gold standard + trait→DOID mapping (reused from the pipeline)

In [3]:
gold_standard = pd.read_pickle(DATA_DIR / 'gold_standard.pkl')
doids_in_gs = set(gold_standard['trait'])
print('gold standard:', gold_standard.shape, '|', len(doids_in_gs), 'DOIDs')

ukb_efo = pd.read_csv(DATA_DIR / 'phenomexcan_traits_fullcode_to_efo.tsv', sep='\t',
                      index_col='ukb_fullcode')
ukb_efo.index = [i.replace('-', '_', 1) for i in ukb_efo.index]
efo_xrefs = pd.read_csv(DATA_DIR / 'term_id_xrefs.tsv.gz', sep='\t')
do_xrefs = pd.read_csv(DATA_DIR / 'xrefs-prop-slim.tsv', sep='\t')

gold standard: (998, 3) | 87 DOIDs


# Drug (L) matrices + per-tissue disease file lists

In [4]:
L = {'gene_based': pd.read_pickle(LINCS_RAW_FILE),
     'module_based_archs4': pd.read_pickle(LINCS_PROJ_FILE)}
DISEASE_FILES = {
    'gene_based': sorted(SPREDIXCAN_RAW_DIR.glob('spredixcan-mashr-zscores-*-data.pkl')),
    'module_based_archs4': sorted(
        SPREDIXCAN_PROJ_DIR.glob('spredixcan-mashr-zscores-*-projection-archs4.pkl')),
}
for m in METHODS:
    print(m, '| L:', L[m].shape, '| tissue files:', len(DISEASE_FILES[m]))
    assert len(DISEASE_FILES[m]) == N_TISSUES

def tissue_of(path):
    return path.stem.replace('spredixcan-mashr-zscores-', '').replace(
        '-projection-archs4', '').replace('-data', '')

gene_based | L: (7120, 1170) | tissue files: 49
module_based_archs4 | L: (1728, 1170) | tissue files: 49


# Helpers: top-N masking, the three per-cell scorings, trait→DOID, NB10 aggregation

In [5]:
def topn_mask(X, n):
    '''Zero all but the top-n entries by abs per column (mirrors _zero_nontop_genes, vectorized).
    Signed values kept. n=None -> no masking.'''
    if n is None or n >= X.shape[0]:
        return X
    absX = np.abs(X)
    # kth largest threshold per column via partition; ties may keep a few extra (negligible)
    kth = np.partition(absX, -n, axis=0)[-n, :]
    return np.where(absX >= kth, X, 0.0)


def score_raw_cosine_pearson(Lc_df, X, drugs):
    '''The three dot-product scorings on the same masked X (k x traits), each drug x trait.
    Normalization ladder (all share the sign convention `-`):
      raw     = -(L_d . x)             keeps norms + mean
      cosine  = -(L_d . x)/(|L_d||x|)  removes norms, keeps mean   (NEXT_STEPS concern #2)
      pearson = -corr(L_d, x)          removes norms + mean        (= cosine of centered vectors)
    Norms/centering use all common genes/LVs (zeros from top-N masking included), matching the
    way pearson is defined here; so cosine and pearson differ only by the mean-centering term.'''
    Lv = Lc_df.values                       # (k x drugs)
    raw = -(Lv.T @ X)                        # (drugs x traits)
    # cosine: scale raw by the L2 norms of each drug and trait vector (no centering)
    Ln = np.sqrt((Lv ** 2).sum(0))[:, None]      # (drugs x 1)
    Xn = np.sqrt((X ** 2).sum(0))[None, :]       # (1 x traits)
    with np.errstate(divide='ignore', invalid='ignore'):
        cos = raw / (Ln * Xn)
    # pearson: cosine of the mean-centered vectors
    Lc = Lv - Lv.mean(axis=0, keepdims=True)
    Xc = X - X.mean(axis=0, keepdims=True)
    num = Lc.T @ Xc
    den = np.sqrt((Lc ** 2).sum(0)[:, None] * (Xc ** 2).sum(0)[None, :])
    with np.errstate(divide='ignore', invalid='ignore'):
        pear = -(num / den)
    return raw, cos, pear


# precompute trait->DOID dict once per method-trait-universe (mirrors map_traits_to_doid)
def build_trait_to_doid(traits):
    doid_efo = efo_xrefs[efo_xrefs['target_id_type'] == 'DOID']
    t2d = {}
    for trait in traits:
        if trait not in ukb_efo.index:
            continue
        rows = ukb_efo.loc[trait]
        if isinstance(rows, pd.Series):
            rows = rows.to_frame().T
        efos = set()
        for tc in rows['term_codes'].dropna():
            for c in str(tc).split(','):
                c = c.strip()
                if c:
                    efos.add(c)
        doids = set()
        for e in efos:
            doids.update(doid_efo[doid_efo['term_id'] == e]['target_id'].values)
            if e.startswith('EFO:'):
                mm = (do_xrefs['resource'] == 'EFO') & (do_xrefs['resource_id'] == e[4:])
                doids.update(do_xrefs[mm]['doid_code'].values)
        if not doids:
            continue
        pref = sorted(doids & doids_in_gs)
        t2d[trait] = pref[0] if pref else sorted(doids)[0]
    return t2d


def to_doid(score_df, t2d):
    '''score_df: drug x trait -> drug x DOID (max over traits mapping to same DOID).'''
    cols = [t for t in score_df.columns if t in t2d]
    s = score_df[cols].rename(columns={t: t2d[t] for t in cols})
    return s.T.groupby(level=0).max().T


def aggregate_nb10(long_df):
    '''mean over thresholds (per trait,drug,method,scoring,tissue) then max over tissues.'''
    g1 = (long_df.groupby(['method', 'scoring', 'trait', 'drug', 'tissue'], observed=True)
          .agg(score=('score', 'mean'), true_class=('true_class', 'first')).reset_index())
    g2 = (g1.groupby(['method', 'scoring', 'trait', 'drug'], observed=True)
          .agg(score=('score', 'max'), true_class=('true_class', 'first')).reset_index())
    return g2

# Recompute raw + pearson + background per (method, tissue, threshold) → merge gold standard

In [6]:
t0 = time.time()
records = []
t2d_cache = {}

for method in METHODS:
    Lm = L[method]
    thresholds = METHOD_THRESHOLDS[method]
    for fpath in tqdm(DISEASE_FILES[method], desc=method, ncols=90):
        tissue = tissue_of(fpath)
        Dt = pd.read_pickle(fpath)
        if not Dt.index.is_unique:
            Dt = Dt[~Dt.index.duplicated(keep='first')]
        common = Lm.index.intersection(Dt.index)
        Lc_df = Lm.loc[common]
        Xdf = Dt.loc[common]
        drugs = list(Lm.columns)

        key = tuple(Xdf.columns)
        if key not in t2d_cache:
            t2d_cache[key] = build_trait_to_doid(Xdf.columns)
        t2d = t2d_cache[key]

        for ntc in thresholds:
            Xm = topn_mask(Xdf.values.astype(float), ntc)
            Xm_df = pd.DataFrame(Xm, index=common, columns=Xdf.columns)
            raw_arr, cos_arr, pear_arr = score_raw_cosine_pearson(Lc_df, Xm, drugs)
            raw_df = pd.DataFrame(raw_arr, index=drugs, columns=Xdf.columns)
            cos_df = pd.DataFrame(cos_arr, index=drugs, columns=Xdf.columns)
            pear_df = pd.DataFrame(pear_arr, index=drugs, columns=Xdf.columns)

            raw_doid = to_doid(raw_df, t2d)        # drug x DOID_full
            cos_doid = to_doid(cos_df, t2d)
            pear_doid = to_doid(pear_df, t2d)
            # background: per-drug z across all DOIDs of the raw full matrix
            mu = raw_doid.mean(axis=1); sd = raw_doid.std(axis=1, ddof=1)
            bg_doid = raw_doid.sub(mu, axis=0).div(sd, axis=0)

            for scoring, sdf in [('raw', raw_doid), ('cosine', cos_doid),
                                 ('pearson', pear_doid), ('background', bg_doid)]:
                long = sdf.copy()
                long.index.name = 'drug'; long.columns.name = 'trait'
                long = long.unstack().reset_index().rename(columns={0: 'score'})
                long['score'] = long['score'].rank()       # rank over full DOID distribution
                long = long.merge(gold_standard, on=['trait', 'drug'], how='inner')
                long['method'] = method; long['scoring'] = scoring
                long['tissue'] = tissue
                records.append(long)

print(f'recompute done in {time.time()-t0:.0f}s; chunks={len(records)}')

gene_based:   0%|                                                  | 0/49 [00:00<?, ?it/s]

gene_based:   2%|▊                                         | 1/49 [00:10<08:33, 10.70s/it]

gene_based:   4%|█▋                                        | 2/49 [00:19<07:30,  9.59s/it]

gene_based:   6%|██▌                                       | 3/49 [00:27<06:55,  9.04s/it]

gene_based:   8%|███▍                                      | 4/49 [00:36<06:41,  8.91s/it]

...

module_based_archs4:  96%|██████████████████████████████▋ | 47/49 [03:45<00:09,  4.84s/it]

module_based_archs4:  98%|███████████████████████████████▎| 48/49 [03:50<00:04,  4.87s/it]

module_based_archs4: 100%|████████████████████████████████| 49/49 [03:55<00:00,  4.87s/it]

module_based_archs4: 100%|████████████████████████████████| 49/49 [03:55<00:00,  4.81s/it]

recompute done in 657s; chunks=1960


# Validation checks (completeness + identical 685-pair universe across methods/scorings)

In [7]:
predictions = pd.concat(records, ignore_index=True)
assert not predictions.isna().any().any()
for c in ['method', 'scoring', 'trait', 'drug', 'tissue']:
    predictions[c] = predictions[c].astype('category')

N_PREDICTIONS = predictions[['drug', 'trait']].drop_duplicates().shape[0]
print('unique (drug, DOID) pairs:', N_PREDICTIONS)
assert N_PREDICTIONS == 685, N_PREDICTIONS

# every method x scoring must cover the SAME 685 pairs (cross-method comparability)
universe = None
for (m, s), g in predictions.groupby(['method', 'scoring'], observed=True):
    pairs = set(map(tuple, g[['drug', 'trait']].drop_duplicates().values))
    n_files = g[['tissue']].drop_duplicates().shape[0]
    assert len(pairs) == 685, (m, s, len(pairs))
    assert n_files == N_TISSUES, (m, s, n_files)
    universe = pairs if universe is None else universe
    assert pairs == universe, (m, s)
print('OK: 49 tissues, identical 685-pair universe across all method x scoring.')

unique (drug, DOID) pairs: 685
OK: 49 tissues, identical 685-pair universe across all method x scoring.


# Aggregate (NB10: mean over thresholds → max over tissues)

In [8]:
aggregated = aggregate_nb10(predictions)
print('aggregated:', aggregated.shape)
aggregated.to_pickle(OUTPUT_DIR / 'predictions_multitissue_aggregated.pkl')
display(aggregated.groupby(['method', 'scoring'], observed=True).size().rename('n_pairs'))

aggregated: (5480, 6)


method               scoring   
gene_based           background    685
                     cosine        685
                     pearson       685
                     raw           685
module_based_archs4  background    685
                     cosine        685
                     pearson       685
                     raw           685
Name: n_pairs, dtype: int64

# Validation: recomputed RAW reproduces the published AUROCs (gene ≈ 0.583, ARCHS4 ≈ 0.625)

In [9]:
raw_auroc = {}
for m in METHODS:
    sub = aggregated[(aggregated.method == m) & (aggregated.scoring == 'raw')]
    raw_auroc[m] = roc_auc_score(sub['true_class'].astype(int), sub['score'])
print('recomputed RAW AUROC:', {k: round(v, 4) for k, v in raw_auroc.items()})
assert abs(raw_auroc['gene_based'] - 0.583) < 0.02, raw_auroc['gene_based']
assert abs(raw_auroc['module_based_archs4'] - 0.625) < 0.02, raw_auroc['module_based_archs4']
print('OK: recompute reproduces the published raw numbers within tolerance.')

recomputed RAW AUROC: {'gene_based': 0.583, 'module_based_archs4': 0.6254}
OK: recompute reproduces the published raw numbers within tolerance.


# Permutation spot-check: NES_permute ≈ pearson under top-N masking (at scale)

Confirms that the analytic Pearson we use across all 245 files is the masked permute-disease null.

In [10]:
rng = np.random.default_rng(SEED)
gate = []
for method in METHODS:
    Lm = L[method]
    fpath = DISEASE_FILES[method][0]            # one representative tissue
    Dt = pd.read_pickle(fpath)
    Dt = Dt[~Dt.index.duplicated(keep='first')]
    common = Lm.index.intersection(Dt.index)
    Lv = Lm.loc[common].values
    sample_traits = list(Dt.columns[:3])        # a few traits
    for ntc in [None, METHOD_THRESHOLDS[method][1]]:   # all-genes and the smallest top-N
        for tr in sample_traits:
            x = topn_mask(Dt.loc[common, [tr]].values.astype(float), ntc)[:, 0]
            # pearson
            Lc = Lv - Lv.mean(0, keepdims=True); xc = x - x.mean()
            den = np.sqrt((Lc ** 2).sum(0) * (xc ** 2).sum())
            with np.errstate(divide='ignore', invalid='ignore'):
                pear = -(Lc.T @ xc) / den
            # permutation NES (permute the masked vector across genes)
            k = x.shape[0]
            idx = np.argsort(rng.random((SPOTCHECK_BPERM, k)), axis=1)
            null = -(Lv.T @ x[idx].T)            # (drugs x B)
            s_obs = -(Lv.T @ x)
            nes = (s_obs - null.mean(1)) / null.std(1, ddof=1)
            ok = np.isfinite(nes) & np.isfinite(pear)
            gate.append({'method': method, 'ntc': ntc, 'trait': tr,
                         'corr': np.corrcoef(nes[ok], pear[ok])[0, 1]})
gate = pd.DataFrame(gate)
display(gate)
assert (gate['corr'] > 0.95).all(), 'NES_permute diverges from pearson under masking'
print('SPOT-CHECK PASSED: pearson == masked permute-disease NES (corr > 0.95) at scale.')

,method,ntc,trait,corr
0,gene_based,NaN,100001_raw_Food_weight,0.996649
1,gene_based,NaN,100002_raw_Energy,0.996524
2,gene_based,NaN,100003_raw_Protein,0.996303
3,gene_based,50.0,100001_raw_Food_weight,0.995692
4,gene_based,50.0,100002_raw_Energy,0.996513
5,gene_based,50.0,100003_raw_Protein,0.994493
6,module_based_archs4,NaN,100001_raw_Food_weight,0.996766
7,module_based_archs4,NaN,100002_raw_Energy,0.997589
8,module_based_archs4,NaN,100003_raw_Protein,0.997456
9,module_based_archs4,5.0,100001_raw_Food_weight,0.994488


SPOT-CHECK PASSED: pearson == masked permute-disease NES (corr > 0.95) at scale.
